# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shakir-j/flyrank-internship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("HF_TOKEN found:", HF_TOKEN is not None)

HF_TOKEN found: True


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
# Section 1 — Build the feature vector

import pandas as pd
import duckdb

# Connect to DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

# Register the Hugging Face token already stored in Colab Secrets
con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# February = feature window
FEB = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/*.parquet'
)
"""

# March = outcome/label window
MAR = """
read_parquet(
    'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
)
"""

# Build February features.
# March is aggregated separately so that the join remains
# at one row per client-content pair.
feature_frame = con.sql(f"""
    WITH feb AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(COALESCE(gsc_impressions, 0)) AS feb_gsc_impressions,
            SUM(COALESCE(gsc_clicks, 0)) AS feb_gsc_clicks,
            AVG(gsc_avg_position) AS feb_gsc_avg_position,
            SUM(COALESCE(ga4_sessions, 0)) AS feb_ga4_sessions,
            SUM(COALESCE(scroll_events, 0)) AS feb_scroll_events

        FROM {FEB}
        GROUP BY client_hash_id, content_hash_id
    ),

    mar AS (
        SELECT
            client_hash_id,
            content_hash_id,

            CASE
                WHEN
                    SUM(COALESCE(gsc_clicks, 0)) > 0
                    OR SUM(COALESCE(ga4_sessions, 0)) > 0
                    OR SUM(COALESCE(scroll_events, 0)) > 0
                THEN 1
                ELSE 0
            END AS march_activity_label

        FROM {MAR}
        GROUP BY client_hash_id, content_hash_id
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,

        f.feb_gsc_impressions,
        f.feb_gsc_clicks,
        f.feb_gsc_avg_position,
        f.feb_ga4_sessions,
        f.feb_scroll_events,

        COALESCE(m.march_activity_label, 0) AS march_activity_label

    FROM feb f
    LEFT JOIN mar m
        ON f.client_hash_id = m.client_hash_id
        AND f.content_hash_id = m.content_hash_id
""").df()

# Model features: IDs are retained only for identification/joining,
# never used as model inputs.
feature_cols = [
    "feb_gsc_impressions",
    "feb_gsc_clicks",
    "feb_gsc_avg_position",
    "feb_ga4_sessions",
    "feb_scroll_events"
]

X = feature_frame[feature_cols].copy()
y = feature_frame["march_activity_label"].astype(int)

print("Feature frame shape:", feature_frame.shape)
print("Features:", feature_cols)
print("Positive label rate:", y.mean())
print()
print(feature_frame.head())

## 2. Feature notes (meaning, missing, categorical, available-when?)

The feature vector uses five February-only behavioral features. GSC impressions measure search visibility, GSC clicks measure search traffic, GSC average position measures search ranking, GA4 sessions measure website sessions, and scroll events measure page interaction. Missing numeric values are left as missing during feature construction and are handled with median imputation in the model pipeline. There are no categorical model features; client and content identifiers are retained only for joins and row identification. All five model features are available in the February window, before the March activity outcome is measured.

In [ ]:
# Section 2 — Verify feature meaning, missing values,
# categorical handling, and availability before prediction

feature_notes = pd.DataFrame({
    "feature": feature_cols,
    "meaning": [
        "February GSC impressions",
        "February GSC clicks",
        "February GSC average position",
        "February GA4 sessions",
        "February scroll events"
    ],
    "dtype": [X[col].dtype for col in feature_cols],
    "missing_values": [X[col].isna().sum() for col in feature_cols],
    "available_before_march": [True] * len(feature_cols),
    "categorical": [False] * len(feature_cols)
})

print(feature_notes)

print("\nMissing values before model imputation:")
print(X[feature_cols].isna().sum())

print("\nCategorical model features:", 0)

## 3. The leakage hunt

I checked the feature vector for possible leakage. The model features are all February-derived metrics, while the prediction label is March activity. I excluded March outcome fields, availability/product flags, and identifiers from the model features because they could reveal information that would not be known at prediction time. The checks below confirm that the selected feature names do not contain label-derived or March fields.

In [ ]:
# Verify the feature vector does not contain obvious leakage fields

import re

# 1. Check for March/label-derived feature names
leakage_keywords = [
    "march",
    "label",
    "activity",
    "outcome",
    "target"
]

suspected_features = [
    col for col in feature_cols
    if any(keyword in col.lower() for keyword in leakage_keywords)
]

# 2. Check that all selected features are February features
non_feb_features = [
    col for col in feature_cols
    if not col.lower().startswith("feb_")
]

# 3. Check that availability/product flags and identifiers are excluded
excluded_fields = [
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available"
]

included_excluded_fields = [
    col for col in excluded_fields
    if col in feature_cols
]

print("Selected features:", feature_cols)
print("Suspected leakage features:", suspected_features)
print("Non-February features:", non_feb_features)
print("Excluded fields accidentally included:", included_excluded_fields)

print()
print("Leakage check passed:",
      len(suspected_features) == 0
      and len(non_feb_features) == 0
      and len(included_excluded_fields) == 0)


## 4. What I excluded and why

I excluded March outcome fields, availability/product flags, and client/content identifiers from the model features because they would not be appropriate predictors at the February prediction point. The final feature vector contains only February-derived metrics. The leakage check found no March, label-derived, or excluded fields among the selected model features.

In [ ]:
# Final verification of the excluded fields

excluded_fields = [
    "client_hash_id",
    "content_hash_id",
    "client_has_gsc",
    "client_has_ga4",
    "gsc_data_available",
    "ga4_data_available",
    "march_activity_label"
]

still_in_features = [
    col for col in excluded_fields
    if col in feature_cols
]

print("Excluded fields found in final feature vector:", still_in_features)
print("Final feature count:", len(feature_cols))
print("Exclusion check passed:", len(still_in_features) == 0)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.